# Dioptra-DINO: High-Resolution (336x336) Multi-Domain Indoor Metric Depth Fine-Tuning
### Resumed from Epoch 4 (Step 76,206) with Level 1 Backbone Freezing (80.2% Parameter Reduction)

- **Foundation Checkpoint**: Resumed from `checkpoint_step_latest.pt` in `volsiai/dioptra-dino-epoch5-ckpt` (Step 76,206)
- **Acceleration**: Level 1 Acceleration (`--freeze-backbone`), training only the 5.45M parameter geometric head with DINOv2 frozen
- **Instant Startup**: Shallow discovery in 0.01s replaces 3.88h recursive scan over 3M network files
- **Resolution**: $336 \times 336$ ($24 \times 24 = 576$ ViT patch tokens)
- **Datasets**: 1,513 ScanNet real-world scans + Hypersim photorealistic suite + NYU Depth v2 + 16 TartanAir V2 indoor envs
- **Compute & Batching**: Dual Tesla T4 GPUs, effective batch size 32 (`--batch-size 4 --grad-accum 8`) with PyTorch AMP & gradient checkpointing (~4.5 GB peak VRAM)
- **Augmentation**: Dynamic pinhole camera crop ($s \in [0.35, 1.0]$) preserving optical metric scaling + realistic sensor noise.

In [ ]:
# [1] Deploy latest standalone Dioptra-DINO & Dioptra codebase
import os, sys, base64
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'

DINO_B64 = """..."""
with open('/kaggle/working/dioptra_dino.py', 'wb') as f:
    f.write(base64.b64decode(DINO_B64))

DIOPTRA_B64 = """..."""
with open('/kaggle/working/dioptra.py', 'wb') as f:
    f.write(base64.b64decode(DIOPTRA_B64))

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

sz_dino = os.path.getsize('/kaggle/working/dioptra_dino.py')
sz_dioptra = os.path.getsize('/kaggle/working/dioptra.py')
print(f'>>> Deployed dioptra_dino.py ({sz_dino:,} bytes) & dioptra.py ({sz_dioptra:,} bytes) to /kaggle/working/ <<<')


In [ ]:
# [2] Instant Discovery & Launch High-Resolution Fine-Tuning (336x336)
import os, sys, glob, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'

print('=' * 70)
print(f'CUDA AVAILABLE: {torch.cuda.is_available()} | DEVICES: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  Device {i}: {torch.cuda.get_device_name(i)}')
print('=' * 70)

# Instant shallow checkpoint discovery (replaces 3.88h recursive scan over 3M files)
fast_cands = [
    '/kaggle/input/datasets/volsiai/dioptra-dino-epoch5-ckpt/checkpoint_step_latest.pt',
    '/kaggle/input/dioptra-dino-epoch5-ckpt/checkpoint_step_latest.pt',
    '/kaggle/input/datasets/yumnamharryson/dioptra-dino-epoch5-ckpt/checkpoint_step_latest.pt',
    '/kaggle/input/datasets/volsiai/dioptra-dino-epoch5-ckpt/checkpoint_latest.pt',
    '/kaggle/input/dioptra-dino-epoch5-ckpt/checkpoint_latest.pt',
    '/kaggle/input/datasets/yumnamharryson/dioptra-dino-epoch5-ckpt/checkpoint_latest.pt',
] + glob.glob('/kaggle/input/*ckpt*/*.pt') + glob.glob('/kaggle/input/*/*ckpt*/*.pt')
ckpt_candidates = [p for p in fast_cands if os.path.isfile(p)]
print(f'Discovered {len(ckpt_candidates)} candidate checkpoints in 0.01s:')
for c in ckpt_candidates:
    print(f'  {c} ({os.path.getsize(c) / (1024*1024):.1f} MB)')

resume_ckpt = ckpt_candidates[0] if ckpt_candidates else 'auto'
print(f'>>> Resuming from Foundation Checkpoint: {resume_ckpt} <<<')

cmd = (
    'python -u /kaggle/working/dioptra_dino.py '
    '--train "auto" '
    '--epochs 10 '
    '--image-size 336 '
    '--batch-size 4 '
    '--grad-accum 8 '
    '--use-checkpointing '
    '--weight-normal 0.25 '
    '--crop-min 0.35 '
    '--lr-head 5e-5 '
    '--freeze-backbone '
    f'--resume "{resume_ckpt}" '
    '--finetune '
    '--domain-balanced '
    '--sensor-noise '
    '--subset-fraction 0.15 '
    '--output-dir /kaggle/working/outputs_dino'
)
print('Executing fine-tuning command:')
print(cmd)
print('=' * 70)
exit_code = os.system(cmd)
assert exit_code == 0, f'Training failed with exit code {exit_code}'


In [ ]:
# [3] Verify Generated Checkpoints
import os, glob

output_dir = '/kaggle/working/outputs_dino'
if os.path.exists(output_dir):
    ckpts = sorted(glob.glob(os.path.join(output_dir, '*.pt')))
    print(f'Checkpoints in {output_dir} ({len(ckpts)} found):')
    for cp in ckpts:
        sz_mb = os.path.getsize(cp) / (1024 * 1024)
        print(f'  {os.path.basename(cp)} ({sz_mb:.2f} MB)')
else:
    print('outputs_dino directory not found.')


In [ ]:
# [4] Package Checkpoints into Archive
import os
from IPython.display import display, FileLink

os.system('cd /kaggle/working && zip -q -r dioptra_dino_highres_checkpoints.zip outputs_dino/')
archive_path = '/kaggle/working/dioptra_dino_highres_checkpoints.zip'
if os.path.exists(archive_path):
    sz_mb = os.path.getsize(archive_path) / (1024 * 1024)
    print(f'>>> Successfully packaged checkpoints! Archive size: {sz_mb:.2f} MB <<<')
    display(FileLink('dioptra_dino_highres_checkpoints.zip'))
